# Neighborhood finder with multiple starting nodes

In [1]:
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder
from TCT import TCT_network_annotator

from TCT import TCT

## Load Translator resources

In [3]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [4]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = [#'Retriever',
                    #'Clinical Trials KP - TRAPI 1.5.0',
                    #'Drug Approvals KP - TRAPI 1.5.0',
                    #'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    #'Microbiome KP - TRAPI 1.5.0',
                    #'MolePro',
                    #'COHD TRAPI',
                    #'RTX KG2 - TRAPI 1.5.0',
                    #'Text Mined Cooccurrence API',
                    #'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX Pharmacogenomics KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
#for api in APInames:
#    if 'Automat' in api and api not in selected_APIlist:
#        selected_APIlist.append(api)
        
#selected_APIlist = ['Retriever'] # select just Retriever endpoint
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}


selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)


All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

## Run Neighborhood finder with multiple inputs

Here, we will run neighborhood_finder_multiple_inputs with multiple gene inputs, and find all drugs that are connected to any of these genes.

In [5]:
IDs = name_resolver.batch_lookup([ 'STAT3',  'MCL1', 'NFKBIA','MKI67','BCL2','BAX','CASP3','STAT5','NFKB1','ATR'], only_taxa='NCBITaxon:9606', biolink_types=['biolink:Gene'])
#IDs = name_resolver.batch_lookup(['MTORC'], only_taxa='NCBITaxon:9606', biolink_types=['biolink:Gene'])

In [6]:
input_identifiers = []
for gene in IDs:
    input_identifiers.append(IDs[gene].curie)
input_identifiers

['NCBIGene:6774',
 'NCBIGene:4170',
 'NCBIGene:4792',
 'NCBIGene:4288',
 'NCBIGene:596',
 'NCBIGene:581',
 'NCBIGene:836',
 'NCBIGene:6777',
 'NCBIGene:4790',
 'NCBIGene:545']

In [7]:
result, result_parsed = TCT_neighborhood_finder.neighborhood_finder_multiple_inputs(input_identifiers,
                                                                                            node2_categories = ['biolink:Gene','biolink:Protein'],
                                                                                            APInames = APInames,
                                                                                            metaKG = metaKG,
                                                                                            API_predicates = API_predicates)   

['NCBIGene:6774', 'NCBIGene:4170', 'NCBIGene:4792', 'NCBIGene:4288', 'NCBIGene:596', 'NCBIGene:581', 'NCBIGene:836', 'NCBIGene:6777', 'NCBIGene:4790', 'NCBIGene:545']
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!


In [8]:
# write a result to a json file
import json
# add a timestamp to the file name
import datetime
# this will create a new file include all the neighbors in the query
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('./PathFinder_testing_results/'+'TCT_neighborhood_finder_result_'+'_'+timestamp+'.json', 'w') as f:
    json.dump(result_parsed, f)

In [9]:
# This will create a new file only include the intermediate nodes with degree > 1  or input nodes in the query, and the edges connecting to them.
result = TCT_network_annotator.get_connected_graph(result_parsed, input_identifiers)

import datetime
import json
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('./PathFinder_testing_results/'+'TCT_neighborhood_finder_result_'+'_'+timestamp+'.json', 'w') as f:
    json.dump(result, f)